# 04 - Results

PRD sections 3.9, 3.12 and 5, flow 4. Every fine-tuned model (3 models x 5 modes) and the **zero-shot Grounding-DINO
baseline** (not trained, class names as text prompt, course 2263) are measured on the **real test split only**.

Per run: mAP@50 (`torchmetrics`), F1 per class at IoU 0.5, recall of small objects, mAP@50 on conditions that never
appeared in the real tune images, **false-approval rate of lookalikes**, **miss rate of forbidden objects**, and the shelter
verdict (accuracy / precision / recall / F1 from Hugging Face `evaluate`). The confidence threshold of each run is chosen on
the control (val) split, never on the test split.

**Read the numbers with the sample size in mind.** The test split has only about 20-30 images: one image moves a
per-image metric by 3-5 points, and subsets (small objects, lookalikes, new conditions) are smaller still. The tables show
the `n_*` behind every rate. A gain that is smaller than one image is not a result.

In [ ]:
%pip install -q transformers accelerate timm torchmetrics pycocotools evaluate scikit-learn

In [ ]:
BRANCH = "dev"  # set to your branch name to run a branch

import os, subprocess, sys, yaml
REPO = "/content/genai-shelter-validator"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-shelter-validator.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))
sys.path.insert(0, f"{REPO}/Code")

from google.colab import drive
drive.mount("/content/drive")
ROOT = cfg["drive_root"]
os.makedirs(ROOT, exist_ok=True)
print("Config loaded, Drive ready at", ROOT)

## 1. Evaluate every fine-tuned model on the real test split only

`New conditions` = test images whose `condition` (field in `real/annotations.json`) never occurs among the real tune images.

In [ ]:
import gc, json
import numpy as np
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModelForObjectDetection, AutoProcessor, AutoModelForZeroShotObjectDetection
import common as C

device = C.device
table = C.load_objects(REPO, cfg)
classes = table["name"].tolist()
K = len(classes)
real = C.split_real(C.load_real(ROOT, classes), cfg, ROOT)
val = [r for r in real if r["split"] == "val"]
test = [r for r in real if r["split"] == "test"]
tune_conditions = {r["condition"] for r in real if r["split"] == "tune"}
new_idx = [i for i, r in enumerate(test) if r["condition"] not in tune_conditions]
print(f"control images: {len(val)} | test images: {len(test)} | test images in new conditions: {len(new_idx)}")
C.check_test_requirements(real, table)

verdict_metrics = evaluate.combine(["accuracy", "precision", "recall", "f1"])
rows, per_class, stored = [], [], {}


def score_run(model_name, mode, preds_val, preds_test):
    thr = C.pick_threshold(preds_val, val, K)
    m = C.run_metrics(preds_test, test, thr, table, cfg)
    m["map50_new_conditions"] = (C.map50([preds_test[i] for i in new_idx], [test[i] for i in new_idx])
                                 if new_idx else float("nan"))
    verdict = verdict_metrics.compute(predictions=m["y_pred"], references=m["y_true"])
    row = {"model": model_name, "mode": mode, "mode_name": C.MODES.get(mode, "zero-shot, no training"),
           **{k: m[k] for k in ["threshold", "map50", "macro_f1", "small_recall", "map50_new_conditions",
                                "lookalike_false_approval", "forbidden_miss", "verdict_false_approval",
                                "n_small", "n_lookalike", "n_forbidden"]},
           "n_new_conditions": len(new_idx), "n_test": len(test), "n_valid_true": int(sum(m["y_true"])),
           **{f"verdict_{k}": v for k, v in verdict.items()}}
    rows.append(row)
    per_class.append({"model": model_name, "mode": mode, **m["f1_per_class"]})
    stored[(model_name, mode)] = (thr, preds_test)

## 2. Off-the-shelf baseline: zero-shot Grounding-DINO

The floor for "fine-tuned versus off-the-shelf": no training, the ten class phrases are the text prompt. It is not one of
the three detection models. A zero-shot model has no way to know what makes a lookalike different from the real object;
expect it to confuse them.

In [ ]:
dino_id = cfg["validation"]["dino_model"]
dino_processor = AutoProcessor.from_pretrained(dino_id)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device).eval()
score_run("Grounding-DINO (zero-shot)", 0, C.predict_zero_shot(dino, dino_processor, val, classes),
          C.predict_zero_shot(dino, dino_processor, test, classes))
del dino
gc.collect()
torch.cuda.empty_cache()
pd.DataFrame(rows)

## 3. Fine-tuned models: 3 models x 5 modes

In [ ]:
for model_id in cfg["detection"]["models"]:
    tag = model_id.split("/")[-1]
    for mode in C.MODES:
        path = f"{ROOT}/results/models/{tag}/mode{mode}"
        if not os.path.exists(f"{path}/run.json"):
            print(f"missing: {tag} mode {mode} (run 03_finetune)")
            continue
        processor = AutoImageProcessor.from_pretrained(path)
        model = AutoModelForObjectDetection.from_pretrained(path)
        score_run(tag, mode, C.predict(model, processor, val), C.predict(model, processor, test))
        del model
        gc.collect()
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
os.makedirs(f"{ROOT}/results/plots", exist_ok=True)
df.to_csv(f"{ROOT}/results/experiments.csv", index=False)
pd.DataFrame(per_class).to_csv(f"{ROOT}/results/f1_per_class.csv", index=False)
show = ["model", "mode_name", "threshold", "map50", "macro_f1", "small_recall", "map50_new_conditions",
        "lookalike_false_approval", "forbidden_miss", "verdict_accuracy", "verdict_precision", "verdict_recall",
        "verdict_f1", "verdict_false_approval"]
display(df[show].round(3))
display(df[["model", "mode_name", "n_test", "n_valid_true", "n_small", "n_new_conditions", "n_lookalike", "n_forbidden"]])

## 4. mAP@50, small-object recall, unseen capture conditions: tables and plots for the slides

One chart per metric, saved to `results/plots/` in Drive (copy the PNG/CSV files you need into `Visuals/` and `Results/`).
Bars are the five modes of each model; the dashed line is the zero-shot baseline. Hatch patterns and value labels repeat
the color coding, so the charts also read in grayscale.

In [ ]:
COLORS = {1: "#2a78d6", 2: "#eb6834", 3: "#1baf7a", 4: "#eda100", 5: "#e87ba4"}
HATCH = {1: "", 2: "//", 3: "\\\\", 4: "xx", 5: ".."}
INK, MUTED = "#0b0b0b", "#52514e"
METRICS = [("map50", "mAP@50", False), ("small_recall", "Recall of small objects", False),
           ("map50_new_conditions", "mAP@50 on unseen capture conditions", False),
           ("lookalike_false_approval", "False-approval rate of lookalikes (lower is better)", True),
           ("forbidden_miss", "Miss rate of forbidden objects (lower is better)", True),
           ("macro_f1", "Macro F1 over classes (IoU 0.5)", False),
           ("verdict_f1", "Shelter verdict: F1 of 'valid'", False),
           ("verdict_false_approval", "Shelter verdict: false approvals (lower is better)", True)]
trained = df[df["mode"] > 0]
model_order = [m.split("/")[-1] for m in cfg["detection"]["models"] if m.split("/")[-1] in set(trained["model"])]
baseline = df[df["mode"] == 0]


def bar_chart(metric, title):
    fig, ax = plt.subplots(figsize=(3.2 * max(len(model_order), 2) + 2, 4.8))
    width = 0.16
    for j, mode in enumerate(C.MODES):
        values = []
        for model in model_order:
            sel = trained[(trained["model"] == model) & (trained["mode"] == mode)][metric]
            values.append(100 * sel.iloc[0] if len(sel) else np.nan)
        xs = np.arange(len(model_order)) + (j - 2) * width
        ax.bar(xs, np.nan_to_num(values), width * 0.88, color=COLORS[mode], hatch=HATCH[mode], edgecolor="white",
               linewidth=0, label=C.MODES[mode])
        for x, v in zip(xs, values):
            ax.text(x, 0 if np.isnan(v) else v, "n/a" if np.isnan(v) else f"{v:.0f}", ha="center", va="bottom",
                    fontsize=8, color=INK)
    if len(baseline) and not np.isnan(baseline[metric].iloc[0]):
        ax.axhline(100 * baseline[metric].iloc[0], color=MUTED, linestyle="--", linewidth=1.5, label="zero-shot Grounding-DINO")
    ax.set_xticks(np.arange(len(model_order)))
    ax.set_xticklabels(model_order, color=INK)
    ax.set_ylabel("%", color=MUTED)
    ax.set_ylim(0, 105)
    ax.set_title(title, loc="left", color=INK)
    ax.grid(axis="y", alpha=0.2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    plt.tight_layout()
    plt.savefig(f"{ROOT}/results/plots/{metric}.png", dpi=200, bbox_inches="tight")
    plt.show()


for metric, title, _ in METRICS:
    bar_chart(metric, title)

## 5. Shelter verdict and confusion between classes

**Verdict (PRD 3.12):** a shelter is *valid* only if every required object is detected above the threshold and no forbidden
object is. The ground truth comes from the labels of the same image. Real images from the web rarely show all five
required objects, so there may be very few (or no) truly valid test images: look at `n_valid_true` above before reading
precision and recall. Compositions of a web shelter plus synthetic objects are demos only and do not enter this table.

**Confusion matrices:** rows are the true class, columns what the model detected on that box (IoU >= 0.5); the last column is
a miss and the last row is a false alarm. The lookalike rows show which models confuse the fake object with the real one.
Mode 1 (real only) is plotted next to mode 4 (real + A + B).

In [ ]:
display(df[["model", "mode_name", "verdict_accuracy", "verdict_precision", "verdict_recall", "verdict_f1",
            "verdict_false_approval", "n_valid_true", "n_test"]].round(3))

labels = classes + ["missed / no object"]
for model in model_order:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
    for ax, mode in zip(axes, [1, 4]):
        if (model, mode) not in stored:
            ax.axis("off")
            continue
        thr, preds = stored[(model, mode)]
        matrix = C.confusion(preds, test, thr, K)
        ax.imshow(matrix, cmap="Blues")
        for (i, j), v in np.ndenumerate(matrix):
            if v:
                ax.text(j, i, int(v), ha="center", va="center", fontsize=8,
                        color="white" if v > matrix.max() / 2 else INK)
        ax.set_xticks(range(K + 1))
        ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(K + 1))
        ax.set_yticklabels(classes + ["no object (false alarm)"], fontsize=8)
        ax.set_xlabel("detected class")
        ax.set_ylabel("true class")
        ax.set_title(f"{model}: {C.MODES[mode]} (threshold {thr:.2f})", loc="left", color=INK)
    plt.tight_layout()
    plt.savefig(f"{ROOT}/results/plots/confusion_{model}.png", dpi=200)
    plt.show()

## 6. Did we reach the PRD section 5 targets?

Differences against mode 1 (real only) for the same model, in percentage points. A target is met if **at least one model**
reaches it with method A, B or A+B (modes 2-4). Mode 5 (classical augmentation) is shown separately: it answers whether
Stable Diffusion adds anything over simple image operations. The last block asks which method helped more and whether the
trend repeats in two models. A negative result is still a result (PRD 5), but only if the numbers are larger than the noise.

In [ ]:
def pp(a, b):
    return 100 * (a - b)


delta = []
for model in model_order:
    base = trained[(trained["model"] == model) & (trained["mode"] == 1)]
    if base.empty:
        continue
    base = base.iloc[0]
    for mode in [2, 3, 4, 5]:
        sel = trained[(trained["model"] == model) & (trained["mode"] == mode)]
        if sel.empty:
            continue
        r = sel.iloc[0]
        delta.append({"model": model, "mode": C.MODES[mode], "d_map50": pp(r["map50"], base["map50"]),
                      "d_small_recall": pp(r["small_recall"], base["small_recall"]),
                      "d_new_conditions": pp(r["map50_new_conditions"], base["map50_new_conditions"]),
                      "d_lookalike_false_approval_drop": pp(base["lookalike_false_approval"], r["lookalike_false_approval"])})
delta = pd.DataFrame(delta)
delta.to_csv(f"{ROOT}/results/delta_vs_real_only.csv", index=False)
display(delta.round(1))

gen_modes = delta[delta["mode"].isin([C.MODES[2], C.MODES[3], C.MODES[4]])]
targets = [("overall: mAP@50 +3 points", "d_map50", 3), ("small objects: recall +5 points", "d_small_recall", 5),
           ("new conditions: +5 points", "d_new_conditions", 5),
           ("lookalikes: false approvals -5 points", "d_lookalike_false_approval_drop", 5)]
for text, col, need in targets:
    best = gen_modes[col].max() if len(gen_modes) else float("nan")
    print(("MET     " if best >= need else "not met"), f"{text}  (best change: {best:+.1f})")

a_vs_b = delta[delta["mode"].isin([C.MODES[2], C.MODES[3]])].pivot(index="model", columns="mode", values="d_map50")
if {C.MODES[2], C.MODES[3]} <= set(a_vs_b.columns):
    better = np.sign(a_vs_b[C.MODES[2]] - a_vs_b[C.MODES[3]])
    print("\nMethod A minus method B in mAP@50 per model (positive = A helped more):")
    display(a_vs_b.assign(A_minus_B=a_vs_b[C.MODES[2]] - a_vs_b[C.MODES[3]]).round(1))
    print("Same winner in at least two models:", bool((better > 0).sum() >= 2 or (better < 0).sum() >= 2))
print(f"\nOne test image is worth about {100 / max(len(test), 1):.1f} points. Compare every difference above with it.")